# AOP-ProCon — Step 2 (build v3): Descriptors + ESM-2 Embeddings
**Component 1 — Fernando W A A T — IT23144408 — J26-DS-302**

Computes the 14 physicochemical descriptors and extracts ESM-2 embeddings (35M / 150M / 650M / 3B) for **every sequence of `AOP-BenchPos_S2.csv`** (Step 1 build v3).
Everything lives under one base path, `BASE = /content/drive/MyDrive/AOP-ProCon` (reads `data/processed`, writes `data/processed` and `data/embeddings`).

## What changed in build v3

| ID | Issue | Fix |
|---|---|---|
| **V3-1** | Step 1 v3 produced a new S2 (**1,389** rows; the earlier S2 had 1,573). The old notebook would have called the existing embeddings "stale" and re-extracted all four scales — hours of compute for nothing. | Embeddings depend **only on the sequence**, so existing files are **re-aligned by sequence** (§0b). Models are loaded only for a scale that is genuinely missing, or when `FORCE_EXTRACT = True`. |
| **V3-2** | The extraction cells ran unconditionally on "Run all". | Every extraction cell is guarded by the plan from §0a. |
| **V3-3** | Hard-coded `1573` in code and text; an assertion of the form `len == 1573 or len == len(aop)` that could never fail. | Counts come from S2. New guards: `homology_group` columns exist, no homology group spans both folds, the S2 fingerprint equals Step 1's. |
| **V3-4** | §4 quoted probe AUROCs (0.809 / 0.804 / 0.806 / 0.796) from an older S2 and older groups, and called the probe "the project's headline result". An independent audit later found ESM-2 within overlapping confidence intervals of a 20-D amino-acid-composition baseline under leak-safe groups. | **All numbers removed.** The scale comparison is computed at run time in Step 5 §13b; §4 now states only the decision rule (E-012). |
| **V3-5** | No record of versions, devices or file hashes; `pip install` unpinned. | `step2_manifest.json` (S2 fingerprint, SHA-256 of every output, versions, device, whether each file was extracted or re-aligned). |
| **V3-6** | Pooling switches from max to mean at 10 aa; external-literature MC peptides (median ≈ 9 aa) and AnOxPePred FRS-only peptides (median 4 aa) therefore sit on different sides of the switch. *Whether this affects results is **not verified**.* | Optional ablation (§2b, off by default): mean-pool at every length for 35M and 150M, so Step 5 can test sensitivity. |

Unchanged and correct: 14 descriptors (GRAVY *is* hydrophobicity), fp32, all four scales, every sequence embedded (GEN included, as Step 5 uses them as queries), and a `_sequences.txt` row-alignment guard beside every `.npy`.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob

BASE = "/content/drive/MyDrive/AOP-ProCon"

# Resolve BASE instead of trusting it: if the expected location has no AOP-BenchPos_S2.csv, search the whole Drive for it.
if not os.path.exists(f"{BASE}/data/processed/AOP-BenchPos_S2.csv"):
    print(f"NOT at the expected BASE: {BASE}")
    if not os.path.exists("/content/drive/MyDrive"):
        raise RuntimeError("Drive is not mounted -- re-run drive.mount() and complete the authorisation prompt.")
    hits = glob.glob("/content/drive/MyDrive/**/AOP-BenchPos_S2.csv", recursive=True)
    if not hits:
        print("My Drive top level:", sorted(os.listdir('/content/drive/MyDrive'))[:40])
        raise FileNotFoundError("AOP-BenchPos_S2.csv not found under My Drive. Run Step 1 first (and mount the account that owns AOP-ProCon).")
    BASE = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    print("resolved BASE ->", BASE)
    if len(hits) > 1:
        print("WARNING: more than one copy exists; using the first:")
        for h in hits: print("   ", h)
print("BASE:", BASE)
!mkdir -p "{BASE}/data/processed" "{BASE}/data/embeddings"

!pip install -q biopython fair-esm

In [ ]:
import os, json, hashlib, shutil, datetime
from importlib import metadata
import numpy as np, pandas as pd, torch
from Bio.SeqUtils.ProtParam import ProteinAnalysis

pd.set_option("display.max_colwidth", 80)
EMB_DIR = f"{BASE}/data/embeddings"
PROC    = f"{BASE}/data/processed"

def _ver(pkg):
    try: return metadata.version(pkg)
    except Exception: return "n/a"
VERSIONS = {"python": __import__("sys").version.split()[0], "torch": torch.__version__, "numpy": np.__version__,
            "pandas": pd.__version__, "biopython": _ver("biopython"), "fair-esm": _ver("fair-esm")}
print(VERSIONS, "| CUDA available:", torch.cuda.is_available())

def s2_fingerprint(s2_df):
    # Canonical content hash of S2 -- identical to the one in Step 1: sequence, labels, homology_group, lso_fold, sample_weight.
    lines = []
    for r in s2_df.sort_values("sequence").itertuples():
        lines.append("\t".join([r.sequence, r.labels,
                                "" if pd.isna(r.homology_group) else str(int(r.homology_group)),
                                "" if pd.isna(r.lso_fold) else str(int(r.lso_fold)),
                                f"{float(r.sample_weight):.1f}"]))
    return hashlib.md5("\n".join(lines).encode()).hexdigest()[:16]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

def save_embeddings(emb, seqs, prefix):
    assert emb.shape[0] == len(seqs), "embedding count != sequence count"
    assert np.isfinite(emb).all(), "NaN/Inf in embeddings"
    np.save(f"{EMB_DIR}/{prefix}.npy", emb)
    with open(f"{EMB_DIR}/{prefix}_sequences.txt", "w") as f:
        f.write("\n".join(seqs))
    print(f"Saved {prefix}: shape={emb.shape} (+ {prefix}_sequences.txt row-alignment guard)")

---
## 0. Load and validate the S2 dataset (Step 1 build v3)
S2 has **one row per unique sequence** with a multi-hot `labels` field. The notebook refuses to continue if Step 1's v3 columns are missing, if any sequence / `cluster_40` id / **homology group**
spans both folds, or if the S2 fingerprint differs from the one Step 1 recorded (set `ALLOW_OTHER_S2 = True` only after deliberately rebuilding Step 1).

In [ ]:
EXPECTED_S2_FINGERPRINT = "47d9c2a2fff63eb1"      # Step 1 build v3 (replay and Colab agree); update it if you deliberately rebuild Step 1

aop = pd.read_csv(f"{PROC}/AOP-BenchPos_S2.csv")

assert aop["sequence"].is_unique, "S2 must be one row per unique sequence"
required = {"sequence", "labels", "sample_weight", "cluster_40", "lso_fold",
            "homology_group", "strict_evidence", "homology_resolvable"}
missing = required - set(aop.columns)
assert not missing, f"AOP-BenchPos_S2.csv is missing {missing} -> it was not produced by Step 1 build v3; run Step 1 first"

S2_SEQS = aop["sequence"].tolist()
N_S2 = len(S2_SEQS)
print("S2 shape:", aop.shape, "| unique sequences:", N_S2)
print(aop["labels"].value_counts().to_dict())
print("FRS-active:", int(aop["labels"].str.contains("FRS").sum()),
      "| MC-active:", int(aop["labels"].str.contains("MC").sum()),
      "| GEN:", int((aop["labels"] == "GEN").sum()))

mech = aop[aop["labels"].str.contains("FRS|MC")]
assert mech["lso_fold"].notna().all() and mech["homology_group"].notna().all(), "every FRS/MC sequence needs a fold and a homology group"
print("mech-active:", len(mech), "| fold0:", int((mech.lso_fold == 0).sum()), "| fold1:", int((mech.lso_fold == 1).sum()))
print("strict-evidence:", int((mech.strict_evidence == True).sum()), "| homology-resolvable (>= 4 aa):", int(mech.homology_resolvable.sum()))

# leakage guards: sequence, CD-HIT cluster AND homology group must each sit in exactly one fold
assert mech.groupby("sequence")["lso_fold"].nunique().max() == 1, "a sequence is in both folds"
assert mech.groupby("cluster_40")["lso_fold"].nunique().max() == 1, "a cluster_40 id is in both folds"
assert mech.groupby("homology_group")["lso_fold"].nunique().max() == 1, "a homology group is in both folds"
print("leak guard: PASS (0 sequences, 0 cluster_40 ids, 0 homology groups span both folds)")

S2_FP = s2_fingerprint(aop)
ALLOW_OTHER_S2 = False       # True only if you deliberately rebuilt Step 1; then also update EXPECTED_S2_FINGERPRINT above
if S2_FP != EXPECTED_S2_FINGERPRINT:
    assert ALLOW_OTHER_S2, (f"S2 fingerprint {S2_FP} != {EXPECTED_S2_FINGERPRINT} (Step 1 build v3): this is another S2. Stopping so that "
                            "embeddings and descriptors are not built for a dataset nobody validated. Set ALLOW_OTHER_S2 = True only if that is intended.")
print("\nS2 fingerprint:", S2_FP, "->", "matches Step 1 build v3" if S2_FP == EXPECTED_S2_FINGERPRINT else "DIFFERENT from Step 1 build v3 (ALLOW_OTHER_S2 = True)")

---
## 0a. Plan: what is already on Drive, and what must be done?
Each embedding file is classified by its `_sequences.txt` guard.

**What a cache check can and cannot prove.** The embedding width identifies the ESM-2 size (480/640/1280/2560), the sequence list proves row alignment, and `isfinite` proves no NaN/Inf. It does **not** prove which checkpoint release, representation layer, pooling rule or precision produced the file. A reused file is therefore recorded in the manifest as *inherited, extraction settings not verified*; the package versions in the manifest are those of **this run**, not of the original extraction. Use `FORCE_EXTRACT = True` if you need every file produced under one known specification.

Classification:

| status | meaning | action |
|---|---|---|
| `ok` | rows are exactly S2's sequences in S2's order | nothing |
| `realign` | the file holds **every** S2 sequence (plus others, or in another order) | slice by sequence in §0b — no model needed |
| `extract` | the file is missing, corrupt, or lacks some S2 sequences | load the model and extract (§2) |

Set `FORCE_EXTRACT = True` to extract every scale from scratch regardless (only useful if you want to rule out the old files).

In [ ]:
FORCE_EXTRACT = False
SCALES = {"35M": 480, "150M": 640, "650M": 1280, "3B": 2560}

def inspect_embedding(scale):
    npy, txt = f"{EMB_DIR}/esm2_{scale}.npy", f"{EMB_DIR}/esm2_{scale}_sequences.txt"
    if not (os.path.exists(npy) and os.path.exists(txt)):
        return "extract", "absent", None
    try:
        arr = np.load(npy, mmap_mode="r")
        seqs = open(txt).read().splitlines()
        finite = bool(np.isfinite(np.asarray(arr)).all())
    except Exception as e:                                   # truncated / corrupt / unreadable cache -> rebuild, do not crash
        return "extract", f"unreadable cache ({type(e).__name__})", None
    if len(seqs) != arr.shape[0] or arr.ndim != 2 or arr.shape[1] != SCALES[scale]:
        return "extract", f"corrupt/unexpected {arr.shape}", arr.shape
    if not finite:
        return "extract", "contains NaN/Inf", arr.shape
    if seqs == S2_SEQS:
        return "ok", "already aligned", arr.shape
    if len(set(seqs)) == len(seqs) and set(S2_SEQS) <= set(seqs):
        return "realign", f"superset of S2 ({len(seqs)} rows)", arr.shape
    return "extract", f"lacks {len(set(S2_SEQS) - set(seqs))} S2 sequences", arr.shape

PLAN = {}
print(f"{'artifact':<24}{'shape':>14}  plan / reason")
for sc in SCALES:
    status, why, shape = inspect_embedding(sc)
    if FORCE_EXTRACT: status, why = "extract", "FORCE_EXTRACT"
    PLAN[sc] = status
    print(f"{'esm2_' + sc + '.npy':<24}{str(shape):>14}  {status:<8} {why}")

_dp = f"{PROC}/descriptors.csv"
if os.path.exists(_dp):
    _d = pd.read_csv(_dp)
    print(f"{'descriptors.csv':<24}{str(_d.shape):>14}  rebuilt in section 1 "
          f"({'already aligned' if _d['sequence'].tolist() == S2_SEQS else 'will be replaced'})")
else:
    print(f"{'descriptors.csv':<24}{'(absent)':>14}  built in section 1")

NEED = {sc: PLAN[sc] == "extract" for sc in SCALES}
EMB_SOURCE = {sc: ("extracted" if NEED[sc] else ("already aligned" if PLAN[sc] == "ok" else "re-aligned from a previous file")) for sc in SCALES}
print("\nmodels will be loaded for:", [sc for sc in SCALES if NEED[sc]] or "none")

---
## 0b. Re-align existing embeddings to S2 *(V3-1)*
An ESM-2 embedding is a function of the sequence alone (the model is fixed and frozen), so it does not depend on labels, folds or on which other sequences are in the dataset.
For each scale marked `realign`, the previous `.npy` is read, the rows are picked **by sequence** in S2's order, and the file is rewritten with its `_sequences.txt`.
The previous pair of files is copied once to `data/embeddings/previous/` and never overwritten. The result is re-read from disk and asserted.

In [ ]:
os.makedirs(f"{EMB_DIR}/previous", exist_ok=True)
for sc, status in PLAN.items():
    if status != "realign":
        continue
    prefix = f"esm2_{sc}"
    old = np.load(f"{EMB_DIR}/{prefix}.npy")
    old_seqs = open(f"{EMB_DIR}/{prefix}_sequences.txt").read().splitlines()
    assert len(old_seqs) == old.shape[0] and len(set(old_seqs)) == len(old_seqs)
    for ext in (".npy", "_sequences.txt"):                               # one-time backup of the previous pair
        dst = f"{EMB_DIR}/previous/{prefix}{ext}"
        if not os.path.exists(dst): shutil.copy(f"{EMB_DIR}/{prefix}{ext}", dst)
    row_of = {s: i for i, s in enumerate(old_seqs)}
    new = old[[row_of[s] for s in S2_SEQS]]
    save_embeddings(new, S2_SEQS, prefix)
    # independent re-read from disk
    chk = np.load(f"{EMB_DIR}/{prefix}.npy")
    assert open(f"{EMB_DIR}/{prefix}_sequences.txt").read().splitlines() == S2_SEQS
    probe = [S2_SEQS[0], S2_SEQS[len(S2_SEQS) // 2], S2_SEQS[-1]]
    assert all(np.array_equal(chk[S2_SEQS.index(s)], old[row_of[s]]) for s in probe), "re-aligned rows differ from the source rows"
    PLAN[sc] = "ok"
    print(f"  {sc}: {old.shape} -> {chk.shape}  re-aligned by sequence; rows verified bit-identical to the source rows")
print("\nplan after 0b:", PLAN)

---
## 1. Physicochemical descriptors — **14 features**

**5 from Biopython `ProteinAnalysis`:** molecular weight, GRAVY, aromaticity, instability index, charge at pH 7.0. The TAF lists "hydrophobicity" and "GRAVY" as two entries; they are the **same computed
value** (`pa.gravy()`), so counting them separately double-counts.

**+1 aliphatic index** — not in Biopython, implemented from Ikai (1980): `AI = X(Ala) + 2.9*X(Val) + 3.9*(X(Ile) + X(Leu))`, `X` = mole percent.

**+8 CTD** — the TAF never enumerates which 8, so this is a documented operationalisation using standard Dubchak group definitions: Hydrophobicity Composition (3) + Transition (3) + Charge Composition
Positive/Negative (2). The charge *neutral* fraction is dropped because Pos + Neu + Neg = 1, so it is exactly recoverable.

**5 + 1 + 8 = 14.**

*Caveat worth stating:* most of these are functions of amino-acid **composition**; the CTD transition features and the instability index additionally depend on adjacent residue pairs (so *not every* anagram pair has an identical vector — the number that do is reported in Step 3). For peptides of median length 5 several features are degenerate, and the instability / aliphatic indices take extreme values on very short sequences. They are a composition-level baseline, not independent evidence. The choice of the 8 CTD features is an implementation decision (the TAF does not enumerate them).

In [ ]:
from itertools import combinations

def aliphatic_index(seq):
    # Ikai (1980): AI = X(Ala) + 2.9*X(Val) + 3.9*(X(Ile)+X(Leu)), X = mole percent
    seq = seq.upper(); L = len(seq)
    return (seq.count("A")/L*100) + 2.9*(seq.count("V")/L*100) \
           + 3.9*((seq.count("I") + seq.count("L"))/L*100)

def compute_core_descriptors(seq):
    pa = ProteinAnalysis(seq)
    return {"mw": pa.molecular_weight(), "gravy": pa.gravy(),
            "aromaticity": pa.aromaticity(), "instability": pa.instability_index(),
            "charge_ph7": pa.charge_at_pH(7.0), "aliphatic_index": aliphatic_index(seq)}

HYDRO_GROUPS  = {"hydrophobic": set("CVLIMFW"), "neutral": set("GASTPHY"), "polar": set("RKEDQN")}
CHARGE_GROUPS = {"positive": set("HKR"), "negative": set("DE")}   # neutral dropped: recoverable

def group_of(aa, groups):
    for name, members in groups.items():
        if aa in members:
            return name
    return None

def ctd_composition(seq, groups):
    L = len(seq); counts = {n: 0 for n in groups}
    for aa in seq:
        g = group_of(aa, groups)
        if g: counts[g] += 1
    return {n: counts[n]/L for n in groups}

def ctd_transition(seq, groups):
    pairs = list(combinations(sorted(groups), 2)); pc = {p: 0 for p in pairs}
    for i in range(len(seq) - 1):
        a, b = group_of(seq[i], groups), group_of(seq[i+1], groups)
        if a and b and a != b:
            pc[tuple(sorted([a, b]))] += 1
    denom = max(len(seq) - 1, 1)
    return {p: c/denom for p, c in pc.items()}

def compute_ctd8(seq):
    seq = seq.upper()
    hc = ctd_composition(seq, HYDRO_GROUPS)
    ht = ctd_transition(seq, HYDRO_GROUPS)
    cc = ctd_composition(seq, CHARGE_GROUPS)
    return {"ctd_hydro_C_hydrophobic": hc["hydrophobic"],
            "ctd_hydro_C_neutral": hc["neutral"],
            "ctd_hydro_C_polar": hc["polar"],
            "ctd_hydro_T_hydrophobic_neutral": ht[("hydrophobic", "neutral")],
            "ctd_hydro_T_neutral_polar": ht[("neutral", "polar")],
            "ctd_hydro_T_hydrophobic_polar": ht[("hydrophobic", "polar")],
            "ctd_charge_C_positive": cc["positive"],
            "ctd_charge_C_negative": cc["negative"]}

# standalone sanity checks on the CTD logic (no Biopython, no Drive)
assert abs(sum(ctd_composition("CCCCCC", HYDRO_GROUPS).values()) - 1.0) < 1e-9
assert ctd_composition("CCCCCC", HYDRO_GROUPS)["hydrophobic"] == 1.0
assert sum(ctd_transition("CCCCCC", HYDRO_GROUPS).values()) == 0.0
assert len(set().union(*HYDRO_GROUPS.values())) == 20, "hydro groups must cover all 20 AA exactly once"
print("CTD logic sanity checks: PASS")

In [ ]:
rows = []
for seq in aop["sequence"]:
    r = {"sequence": seq}
    r.update(compute_core_descriptors(seq))
    r.update(compute_ctd8(seq))
    rows.append(r)

descriptors = pd.DataFrame(rows)
feature_cols = [c for c in descriptors.columns if c != "sequence"]

print("descriptors:", descriptors.shape, "->", len(feature_cols), "features + sequence")
print(feature_cols)

# checkpoint: one row per S2 sequence, same order, no duplicates, no NaN
assert len(feature_cols) == 14, f"expected 14 features, got {len(feature_cols)}"
assert len(descriptors) == len(aop), "descriptor row count != S2 row count"
assert descriptors["sequence"].is_unique, "duplicate sequence rows"
assert (descriptors["sequence"].values == aop["sequence"].values).all(), "row order mismatch vs S2"
assert descriptors.isna().sum().sum() == 0, "NaN in descriptors"

descriptors.to_csv(f"{PROC}/descriptors.csv", index=False)
print(f"\nSaved descriptors.csv -- {descriptors.shape}, unique, row-aligned to AOP-BenchPos_S2.csv")
descriptors.describe().T[["min", "max"]]

---
## 2. ESM-2 embeddings — all four scales, fp32, every sequence

**Pooling (the pipeline's rule):** per-residue embeddings are **max-pooled for < 10 aa and mean-pooled for ≥ 10 aa** (this is a design choice, not a measured result: nothing in this notebook shows that mean-pooling is worse for short peptides; the median length is 5).
This introduces a representation switch at 10 aa, which is a possible confound between length and representation — the mean-pool-everywhere ablation in §2b is the test, and until it has been run the pooling choice must not be described as validated.

**Why every sequence, including GEN:** Step 5 uses GEN peptides as *query* sequences against the trained prototypes, so they must be embedded.

**Why all four scales:** the scale ablation (HP5) needs them side by side. Embeddings are keyed by **sequence only**, so they are valid for any S2 built from the same sequences.

**Precision deviation, stated explicitly:** the TAF specifies fp16 (a VRAM measure for a local RTX 4050 6 GB target). This runs **fp32** — the Colab hosts fit it, and fp32 is strictly higher precision.

**Only the scales marked `extract` in §0a are run below**; the others are skipped with a message.

In [ ]:
import esm

def load_esm_model(model_fn_name):
    model, alphabet = getattr(esm.pretrained, model_fn_name)()
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return model.to(device), alphabet, device

def embed_sequences(seqs, model, alphabet, device, batch_size=32, pooling="auto"):
    # pooling="auto": max-pool for < 10 aa, mean-pool for >= 10 aa (the pipeline's rule).
    # pooling="mean": mean-pool for EVERY length (sensitivity ablation, section 2b).
    assert pooling in ("auto", "mean")
    bc = alphabet.get_batch_converter(); nl = model.num_layers; out = []
    for start in range(0, len(seqs), batch_size):
        chunk = seqs[start:start + batch_size]
        _, _, tokens = bc([(str(i), s) for i, s in enumerate(chunk)])
        tokens = tokens.to(device)
        with torch.no_grad():
            reps = model(tokens, repr_layers=[nl])["representations"][nl]
        for i, s in enumerate(chunk):
            per_residue = reps[i, 1:len(s) + 1].cpu()
            use_max = (pooling == "auto") and len(s) < 10
            out.append((per_residue.max(0).values if use_max else per_residue.mean(0)).numpy())
        print(f"  {min(start + batch_size, len(seqs))}/{len(seqs)}", end="\r")
    print()
    return np.stack(out)

all_seqs = S2_SEQS
print("sequences to embed (if any scale needs it):", len(all_seqs))

### 35M — sanity test on 20 sequences, then the full pool (only if `extract`)

In [ ]:
if NEED["35M"]:
    test_seqs = all_seqs[:20]
    model, alphabet, device = load_esm_model("esm2_t12_35M_UR50D")
    print("Device:", device, "| layers:", model.num_layers, "| embed_dim:", model.embed_dim)
    test_emb = embed_sequences(test_seqs, model, alphabet, device)
    assert test_emb.shape == (20, model.embed_dim)      # asserted against the model's OWN live embed_dim
    assert np.isfinite(test_emb).all()
    print("Sanity test passed:", test_emb.shape)
    emb = embed_sequences(all_seqs, model, alphabet, device)
    save_embeddings(emb, all_seqs, "esm2_35M")
    del model
    if device == "cuda": torch.cuda.empty_cache()
    PLAN["35M"] = "ok"
else:
    print("35M: embeddings already aligned to S2 (", EMB_SOURCE["35M"], ") -> extraction skipped")

### 150M — only if `extract`

In [ ]:
if NEED["150M"]:
    model, alphabet, device = load_esm_model("esm2_t30_150M_UR50D")
    print("Device:", device)
    emb = embed_sequences(all_seqs, model, alphabet, device)
    save_embeddings(emb, all_seqs, "esm2_150M")
    del model
    if device == "cuda": torch.cuda.empty_cache()
    PLAN["150M"] = "ok"
else:
    print("150M: embeddings already aligned to S2 (", EMB_SOURCE["150M"], ") -> extraction skipped")

### 650M — only if `extract`  *(the TAF's specified primary backbone)*

In [ ]:
if NEED["650M"]:
    model, alphabet, device = load_esm_model("esm2_t33_650M_UR50D")
    print("Device:", device)
    emb = embed_sequences(all_seqs, model, alphabet, device, batch_size=16)
    save_embeddings(emb, all_seqs, "esm2_650M")
    del model
    if device == "cuda": torch.cuda.empty_cache()
    PLAN["650M"] = "ok"
else:
    print("650M: embeddings already aligned to S2 (", EMB_SOURCE["650M"], ") -> extraction skipped")

### 3B — only if `extract`
`batch_size=8`. **Known compute constraint, documented:** on free-tier Colab the T4 runtime disconnects while *loading* the 3B checkpoint — the limit is **host System RAM (~12–13 GB)**, not GPU VRAM,
because `torch.load` pulls the whole fp32 state dict into CPU RAM first. Workaround used earlier: switch to the free `v5e-1 TPU` runtime purely for its larger host VM (47 GB) and let the code fall back to CPU
(`torch.cuda.is_available()` is `False` there). **The TPU chip is never used.** With §0b, the 3B file is normally just re-aligned and this constraint does not arise.

In [ ]:
if NEED["3B"]:
    model, alphabet, device = load_esm_model("esm2_t36_3B_UR50D")
    print("Device:", device)
    emb = embed_sequences(all_seqs, model, alphabet, device, batch_size=8)
    save_embeddings(emb, all_seqs, "esm2_3B")
    del model
    if device == "cuda": torch.cuda.empty_cache()
    PLAN["3B"] = "ok"
else:
    print("3B: embeddings already aligned to S2 (", EMB_SOURCE["3B"], ") -> extraction skipped")

### 2b. Optional ablation — mean-pool at every length *(V3-6; off by default)*
The pipeline max-pools below 10 aa and mean-pools from 10 aa. External-literature MC peptides are longer (median ≈ 9 aa, walnut ≈ 13) than AnOxPePred FRS-only peptides (median 4 aa), so the two
classes sit on different sides of that switch. **Whether this changes any result is not verified.** Setting `RUN_POOLING_ABLATION = True` also writes `esm2_35M_meanall.npy` and `esm2_150M_meanall.npy`
(mean-pool for every sequence) so Step 5 can repeat the probe on them and report whether the conclusions depend on the pooling rule.

In [ ]:
RUN_POOLING_ABLATION = False
if RUN_POOLING_ABLATION:
    for sc, fn in (("35M", "esm2_t12_35M_UR50D"), ("150M", "esm2_t30_150M_UR50D")):
        model, alphabet, device = load_esm_model(fn)
        emb = embed_sequences(all_seqs, model, alphabet, device, pooling="mean")
        save_embeddings(emb, all_seqs, f"esm2_{sc}_meanall")
        del model
        if device == "cuda": torch.cuda.empty_cache()
else:
    print("pooling ablation skipped (RUN_POOLING_ABLATION = False)")

---
## 3. Verify every saved artifact from disk *(not from memory)*
Reloads each `.npy` and its `_sequences.txt` and re-asserts row-alignment against `AOP-BenchPos_S2.csv`, independently of the cells above. Ablation files, if present, are checked too.

In [ ]:
n = len(aop)
print(f"{'file':<22}{'shape':>14}{'dim ok':>8}{'row-align':>11}{'NaN':>6}")
files = {f"esm2_{sc}": dim for sc, dim in SCALES.items()}
for sc in ("35M", "150M"):
    if os.path.exists(f"{EMB_DIR}/esm2_{sc}_meanall.npy"):
        files[f"esm2_{sc}_meanall"] = SCALES[sc]
for prefix, dim in files.items():
    e = np.load(f"{EMB_DIR}/{prefix}.npy")
    seqs = open(f"{EMB_DIR}/{prefix}_sequences.txt").read().splitlines()
    ok_dim, ok_align, ok_nan = (e.shape == (n, dim)), (np.array(seqs) == aop["sequence"].values).all(), bool(np.isfinite(e).all())
    print(f"{prefix:<22}{str(e.shape):>14}{str(ok_dim):>8}{str(bool(ok_align)):>11}{str(ok_nan):>6}")
    assert ok_dim and ok_align and ok_nan, f"{prefix} failed verification"

d = pd.read_csv(f"{PROC}/descriptors.csv")
assert d.shape[1] == 15 and d["sequence"].is_unique and (d["sequence"].values == aop["sequence"].values).all()
print(f"\ndescriptors.csv {d.shape}: unique, row-aligned, 14 features  OK")
print("All Step 2 artifacts verified from disk.")

### 3b. Manifest *(V3-5)*
Records the S2 fingerprint, the SHA-256 of every output, library versions, the device, and — per scale — whether the file was **extracted** or **re-aligned** from a previous file. Steps 3–5 can assert
`s2_fingerprint` against this file to prove they are working on the S2 these embeddings belong to.

In [ ]:
outs = ["descriptors.csv"] + [f"esm2_{sc}.npy" for sc in SCALES]
MODEL_NAMES = {"35M": "esm2_t12_35M_UR50D", "150M": "esm2_t30_150M_UR50D", "650M": "esm2_t33_650M_UR50D", "3B": "esm2_t36_3B_UR50D"}
def _prov(sc):
    if EMB_SOURCE[sc] == "extracted":
        return {"status": "extracted in this notebook", "model": MODEL_NAMES[sc], "layer": "last", "pooling": "max <10 aa / mean >=10 aa", "precision": "fp32",
                "versions": VERSIONS, "device": "cuda" if torch.cuda.is_available() else "cpu"}
    return {"status": "inherited from an earlier run; extraction settings NOT verified (only width = ESM-2 size, row order and finiteness were checked)",
            "assumed_model": MODEL_NAMES[sc], "versions_of_original_extraction": "unknown"}
manifest2 = {
    "run_date": datetime.date.today().isoformat(),
    "s2_fingerprint": S2_FP, "s2_rows": N_S2,
    "versions_this_run": VERSIONS, "cuda_this_run": bool(torch.cuda.is_available()),
    "extraction_provenance": {sc: _prov(sc) for sc in SCALES},
    "sequence_list_sha256": {sc: sha256_file(f"{EMB_DIR}/esm2_{sc}_sequences.txt") for sc in SCALES},
    "pooling_rule": "max-pool < 10 aa, mean-pool >= 10 aa (ablation files *_meanall use mean-pool for every length)",
    "embedding_source": EMB_SOURCE,
    "sha256": {f: sha256_file(f"{PROC}/{f}" if f.endswith(".csv") else f"{EMB_DIR}/{f}") for f in outs},
}
# keep the ORIGIN of a file if this is a re-run and the file is byte-identical to the one the previous manifest described
_mp = f"{PROC}/step2_manifest.json"
if os.path.exists(_mp):
    _prev = json.load(open(_mp))
    for sc in SCALES:
        if EMB_SOURCE[sc] == "already aligned" and _prev.get("sha256", {}).get(f"esm2_{sc}.npy") == manifest2["sha256"][f"esm2_{sc}.npy"]:
            EMB_SOURCE[sc] = _prev["embedding_source"].get(sc, EMB_SOURCE[sc])
            if _prev.get("extraction_provenance", {}).get(sc): manifest2["extraction_provenance"][sc] = _prev["extraction_provenance"][sc]
    manifest2["embedding_source"] = EMB_SOURCE
json.dump(manifest2, open(_mp, "w"), indent=2)
print(json.dumps({k: manifest2[k] for k in ("s2_fingerprint", "s2_rows", "embedding_source")}, indent=2))
print("Saved step2_manifest.json  (assert manifest['s2_fingerprint'] == s2_fingerprint(S2) at the top of Steps 3-5)")

---
## 4. Which backbone is "primary"? *(E-012 — the old numbers are removed)*

**What earlier versions said:** that 150M should replace 650M, from a Wilcoxon test on a *saturated* retrieval metric (35M→150M p = 0.0094, 150M→650M p = 0.045, 650M→3B p = 0.059). p = 0.045 does **not** survive
this project's own Bonferroni correction (α = 0.05 / 7 ≈ 0.0071), so that recommendation was retracted. A later version replaced it with probe AUROCs — those numbers came from an older S2 and from groups that
did not cover near-duplicate short peptides, and they are **no longer quoted here**.

**What to do instead.** The scale comparison is computed **at run time** in Step 5 §13b (supervised probe, out-of-fold, bootstrap 95 % CI, grouped by `homology_group`). Nothing is hard-coded in this notebook.

**Decision rule (stated now, applied there):**
* **650M stays the TAF-specified backbone** for the contrastive models.
* **Overlapping confidence intervals are not evidence that two scales are equivalent** (they only show that the intervals cannot separate them at this sample size). The comparison must be **paired**: the same held-out queries scored by every scale, and the *difference* in the metric with its own bootstrap 95 % CI. "Equivalent" may be claimed only against a **margin fixed before looking at the data** (for example ΔAUROC within ±0.02) and only if the whole CI of the difference lies inside it. Otherwise the honest statement is "no difference detected".
* Choosing the smallest backbone is an **efficiency decision**, taken independently of that comparison and labelled as such.
* Any claim must also be read against a **20-D amino-acid-composition baseline**: an independent audit (2026-10-16) found ESM-2 and that baseline within overlapping confidence intervals under leak-safe groups. Do not claim
  more than Step 5 shows, and do not call the probe "the project's headline result" unless it clears that baseline.

---
## Summary

| Output | Shape | Path |
|---|---|---|
| Physicochemical descriptors (14 features) | (N, 15) | `data/processed/descriptors.csv` |
| ESM-2 35M / 150M / 650M / 3B | (N, 480 / 640 / 1280 / 2560) | `data/embeddings/esm2_<scale>.npy` |
| Optional mean-pool-all ablation | (N, 480 / 640) | `data/embeddings/esm2_<scale>_meanall.npy` |
| Manifest | — | `data/processed/step2_manifest.json` |

`N` = the number of rows of `AOP-BenchPos_S2.csv` (Step 1 build v3: 1,389). Each `.npy` has a companion `_sequences.txt` recording the exact row order.

**Next: Step 3** — positive-pair construction under the "share ≥ 1 mechanism label" rule (re-run with `OVERWRITE = True`; the pools changed).